# ДЗ 2. Обучение LLM: геометрия шага и цена оптимизатора

Фамилия, имя:  
GPU и дата запуска:

Работайте в Colab с GPU. Весь ваш код и ответы остаются в этом ноутбуке.
Измерения и сохранение результатов вынесены в предоставленные файлы.
Перед обучением проверьте
лосс и коллатор на коротких тестах. Папка завершённого запуска не
перезаписывается: после изменения кода смените `RUN_TAG`.

## 0. Окружение

Скачайте [ноутбук задания](https://github.com/BRAIn-Lab-teaching/ADDITIONAL-CHAPTERS-NLP-COURSE/blob/main/hw02/report.ipynb)
и откройте его в Colab через **Файл → Загрузить блокнот**. Выберите
среду с GPU T4. Ячейка ниже скачает репозиторий с проверками,
инфраструктурой и кодом графиков; если он уже скачан, выполнит `git pull`.
Ваш код и ответы остаются в загруженном ноутбуке, а не в клоне репозитория.

Зависимости используйте из настроенной среды: ноутбук ничего не устанавливает.
Задание рассчитано на Colab и NVIDIA T4:
ориентируйтесь на 12 ГБ доступной памяти, даже если карта сообщает больше.

In [ ]:
from pathlib import Path
import subprocess
import sys

repository = Path("/content/ADDITIONAL-CHAPTERS-NLP-COURSE")
if repository.exists():
    subprocess.run(["git", "-C", str(repository), "pull", "--ff-only"], check=True)
else:
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/BRAIn-Lab-teaching/ADDITIONAL-CHAPTERS-NLP-COURSE.git",
        str(repository),
    ], check=True)
%cd $repository
sys.path.insert(0, str(repository))

In [ ]:
import gc
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

from hw02.infra import seed_everything
from hw02.notebook_version.checks import (
    check_apollo_mini, check_causal_collator, check_causal_loss,
    check_geometry, check_parameter_groups,
)
from hw02.notebook_version.visuals import (
    plot_quality, plot_memory, plot_newton_schulz, plot_geometry,
    plot_checkpointing,
)
from hw02.notebook_version.muon import Muon
from hw02.notebook_version.trainer_runtime import run_experiment

ROOT = Path("hw02/notebook_version")
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
SEED = 42
MAX_LENGTH = 256
MIN_LENGTH = 64
TOTAL_STEPS = 180
NS_STEPS = 5
device = torch.device("cuda")
print(torch.cuda.get_device_name(device))
print("Доступно, ГБ:", round(torch.cuda.mem_get_info(device)[0] / 2**30, 2))

## 1. Causal loss

Для логитов формы $[B,T,V]$ и меток $[B,T]$ верните **сумму** cross-entropy
по незамаскированным таргет-токенам и их число скалярным тензором. Готовый
`model(..., labels=...)` здесь не используйте. Подумайте, почему последний
вектор логитов и первая метка не образуют обучающую пару.

Метки со значением `-100` не участвуют в лоссе. Это служебное значение
`ignore_index` в `torch.nn.functional.cross_entropy`, а не ID токена.
Исключайте такие метки и из суммы лосса, и из числа таргет-токенов.

Эта функция нужна только для упражнения и тестов. Во всех дальнейших
запусках используется встроенный лосс модели и обычный `Trainer`.

In [ ]:
def causal_lm_loss(logits: torch.Tensor, labels: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    # Верните (loss_sum, число таргет-токенов как скалярный тензор).
    raise NotImplementedError

check_causal_loss(causal_lm_loss)
print("causal loss: проверка пройдена")

**Вопрос:** В чём проблема усреднения лосса по отдельным микробатчам
при gradient accumulation, если в них разное число таргет-токенов?
Как правильно нормировать лосс по эффективному батчу?

<!-- ответ -->

## 2. Данные, модель и Trainer

Подготовьте данные и запустите обучение с помощью экосистемы HF.
Нужные функции указаны в ячейках; разберитесь с их аргументами и
форматами данных. Реализуйте батчированную токенизацию, примените её через
`Dataset.map`, соберите батч через `tokenizer.pad` и настройте `Trainer`.

Используем SmolLM2-135M в версии **base**: она обучена предсказывать
следующий токен, а не выполнять инструкции. SFT и различия версий
моделей разберём на следующей лекции. У этой версии нет шаблона диалога;
WikiText — обычный текст, поэтому `apply_chat_template` здесь не нужен.

Каждый документ заканчивается EOS и занимает не более 256 токенов.

In [ ]:
seed_everything(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# У base-версии SmolLM2 нет PAD. Использовать EOS для заполнения батча —
# обычный приём для causal LM; настоящий EOS отличаем через attention_mask.
tokenizer.pad_token = tokenizer.eos_token

raw = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
raw_train = raw["train"].select(range(80_000))
raw_eval = raw["validation"].select(range(512))

def tokenize_batch(batch):
    # batch["text"] — список текстов.
    # Без padding и автоматических special tokens; усечение до MAX_LENGTH - 1.
    encoded = tokenizer(batch["text"], ...)
    input_ids = ...  # Добавьте EOS к каждому списку из encoded["input_ids"].
    return {"input_ids": input_ids}

def prepare_split(split):
    split = split.filter(lambda row: bool(row["text"].strip()))
    split = split.map(
        tokenize_batch,
        batched=...,
        remove_columns=...,
    )
    return split.filter(lambda row: len(row["input_ids"]) >= MIN_LENGTH)

train_data = prepare_split(raw_train)
eval_data = prepare_split(raw_eval)
print("Документов train/validation:", len(train_data), len(eval_data))
print("Первые длины:", [len(train_data[i]["input_ids"]) for i in range(5)])

Коллатор получает список строк вида `{"input_ids": [...]}`. Используйте
`tokenizer.pad`: он заполнит батч справа и построит `attention_mask`.
Выберите режим padding и получите тензоры PyTorch. Постройте `labels`
как копию `input_ids`. В метках ставьте `-100` только на местах
добавленного PAD. Настоящий EOS остаётся
таргетом. Первую метку специально маскировать не нужно: её исключает
causal shift во встроенном лоссе модели.

In [ ]:
def collate_causal(rows):
    batch = tokenizer.pad(rows, padding=..., return_tensors=...)
    batch["labels"] = ...  # Копия input_ids.
    batch["labels"][...] = -100  # Маска добавленного PAD через attention_mask.
    return batch

check_causal_collator(collate_causal, pad_token_id=tokenizer.pad_token_id)
example_loader = DataLoader(train_data, batch_size=4, shuffle=False, collate_fn=collate_causal)
example = next(iter(example_loader))
print({name: tuple(value.shape) for name, value in example.items()})
print("Таргетов в примере:", int((example["labels"][:, 1:] != -100).sum()))

In [ ]:
def new_model(checkpointing: bool):
    model = AutoModelForCausalLM.from_pretrained(..., dtype=torch.float32)
    # KV-cache нужен при пошаговой генерации, но не при обучении полных
    # последовательностей; с gradient checkpointing его выключают.
    model.config.use_cache = False
    if checkpointing:
        model.gradient_checkpointing_enable()
    return model

model = new_model(False)
print("Параметров:", sum(p.numel() for p in model.parameters()))
print(model)

In [ ]:
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape))

Выполните шесть пробных шагов обычного `Trainer` со встроенным лоссом
модели. Передайте подготовленный датасет и свой коллатор. Causal shift
уже выполняется внутри модели: сдвигать `labels` самостоятельно нельзя.

Для этого запуска используйте микробатч 2 и накопление по 2 микробатчам.
`fp16=True` включает mixed precision для T4. `Trainer` сам создаёт
нужный `GradScaler` и выполняет gradient accumulation; вручную их настраивать
не нужно.

In [ ]:
args = TrainingArguments(
    output_dir=str(ROOT / "trainer_sanity"),
    per_device_train_batch_size=..., gradient_accumulation_steps=...,
    max_steps=6, learning_rate=5e-5, logging_steps=1,
    save_strategy="no", report_to="none", remove_unused_columns=False,
    fp16=True, bf16=False,
    disable_tqdm=True,
)
trainer = Trainer(
    model=..., args=..., train_dataset=..., data_collator=...,
)
trainer.train()
del trainer, model
gc.collect()
torch.cuda.empty_cache()

**Вопрос:** Почему нельзя маскировать метки по равенству
`pad_token_id`? С каким токеном сравнивается первый вектор логитов?
Чем отличаются батч, который принимает `Dataset.map`, и батч,
который возвращает коллатор?

<!-- ответ -->

## 3. Группы параметров

Допишите условие `is_matrix` в шаблоне ниже. Оно должно отделить веса
скрытых `nn.Linear` от остальных обучаемых параметров. Первая группа
пойдёт в Muon или APOLLO-Mini, вторая — в AdamW.

Опирайтесь на распечатку имён и форм параметров из предыдущего раздела.
В шаблоне `excluded` содержит `id` параметров embedding и выходного слоя,
а `linear_weights` — `id` весов всех `nn.Linear`.

Одна проверка `ndim == 2` недостаточна: embedding тоже матрица.
В SmolLM2 входные embedding и выходной слой используют один
`Parameter` (tied weights). `model.named_parameters()` возвращает
такой параметр один раз; его нельзя включать в обе группы.

In [ ]:
def split_parameters(model):
    excluded = {id(parameter)
                for module in (model.get_input_embeddings(), model.get_output_embeddings())
                for parameter in module.parameters()}
    linear_weights = {id(module.weight) for module in model.modules()
                      if isinstance(module, nn.Linear)}
    matrices, others = [], []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        is_matrix = ...  # Какие веса из linear_weights не входят в excluded?
        destination = matrices if is_matrix else others
        destination.append((name, parameter))
    return matrices, others

group_model = new_model(False)
check_parameter_groups(group_model, split_parameters)
matrix_group, other_group = split_parameters(group_model)
print("Скрытые матрицы:", len(matrix_group), "остальные:", len(other_group))
print("Примеры:", [name for name, _ in matrix_group[:5]])
del group_model
gc.collect()

**Вопрос:** Почему embedding, выходной слой и нормы не идут
в Muon? Что изменится, если веса выходного слоя перестанут быть tied?

<!-- ответ -->

## 4. APOLLO-Mini

В [APOLLO-Mini](https://proceedings.mlsys.org/paper_files/paper/2025/file/437bc4ccafd3fc6d4289bd10940be42b-Paper-Conference.pdf)
моменты Adam хранятся для проекции градиента на один случайный вектор.
Из них получается один масштаб для полного матричного градиента.
Для матрицы $m\times n$ обычный Adam хранит $2mn$ элементов в двух
моментах. Здесь длина проекции — $\min(m,n)$, а длина каждого момента —
$\max(m,n)$. Вместе с двумя скалярами это

$$
\min(m,n)+2\max(m,n)+O(1)=O(m+n).
$$

Это память состояния оптимизатора, а не всей модели: веса и полный
градиент остаются матрицами $m\times n$.

В статье проекцию пересэмплируют каждые 200 шагов. Наш запуск короче:
180 шагов. Поэтому в этом задании достаточно создать её при первом
обновлении; пересэмплирование писать не нужно.

Реализуйте `step` по формулам ниже. Рассмотрен случай $m\le n$ и
проекция слева, как в Algorithm 1 статьи. При $m>n$ сначала
транспонируем градиент, поэтому формулы остаются такими же.

Для градиента $G_t\in\mathbb R^{m\times n}$ и фиксированного вектора
$p\in\mathbb R^m$, чьи компоненты выбраны из $\mathcal N(0,1)$:

$$
R_t=p^\top G_t.
$$

Начните с нулевых моментов и на каждом шаге вычисляйте

$$
M_t=\beta_1M_{t-1}+(1-\beta_1)R_t,
\qquad
V_t=\beta_2V_{t-1}+(1-\beta_2)R_t^2.
$$

$$
\widehat M_t=\frac{M_t}{1-\beta_1^t},
\qquad
\widehat V_t=\frac{V_t}{1-\beta_2^t},
\qquad
Q_t=\frac{\widehat M_t}{\sqrt{\widehat V_t}+\varepsilon}.
$$

$$
s_t=\frac{\|Q_t\|_2}{\|R_t\|_2+10^{-8}}.
$$

Ограничим рост нормы обновления множителем 1.01. На первом шаге
$c_1=1$; далее, если $d_{t-1}$ — сохранённая норма прошлого обновления:

$$
c_t=\max\left(1,\frac{\|s_tG_t\|_F}{1.01(d_{t-1}+10^{-8})}\right),
\qquad
d_t=\frac{\|s_tG_t\|_F}{c_t}.
$$

Итоговый шаг с decoupled weight decay; в коде число 128 задаётся
аргументом `scale`:

$$
W_t=(1-\eta\lambda)W_{t-1}
     -\eta\sqrt{128}\,\frac{s_t}{c_t}G_t.
$$

Инициализация состояния, обход параметров, ограничение роста нормы
и decoupled weight decay уже написаны. Замените многоточия в восьми
строках: вычислите проекцию градиента, обновите моменты, выполните
коррекцию смещения, найдите масштаб и обновите веса.
Полноразмерные моменты и копию градиента в состоянии не храните.

In [ ]:
class ApolloMini(torch.optim.Optimizer):
    def __init__(self, params, *, lr, betas=(0.9, 0.95), eps=1e-6,
                 weight_decay=0.1, scale=128.0, seed=42):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,
                                      weight_decay=weight_decay, scale=scale, seed=seed))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        matrix_index = 0
        for group in self.param_groups:
            beta1, beta2 = group["betas"]
            for parameter in group["params"]:
                index = matrix_index
                matrix_index += 1
                if parameter.grad is None:
                    continue
                gradient = parameter.grad
                # В формулах работаем только со случаем m <= n.
                G = gradient.T if gradient.shape[0] > gradient.shape[1] else gradient
                m, n = G.shape
                state = self.state[parameter]
                if not state:
                    generator = torch.Generator(device=parameter.device).manual_seed(group["seed"] + index)
                    state["projection"] = torch.randn(
                        m, device=parameter.device, dtype=parameter.dtype, generator=generator
                    )
                    state["exp_avg"] = torch.zeros(n, device=parameter.device, dtype=parameter.dtype)
                    state["exp_avg_sq"] = torch.zeros_like(state["exp_avg"])
                    state["step"] = 0

                state["step"] += 1
                projected = ...
                state["exp_avg"].lerp_(..., ...)
                state["exp_avg_sq"].mul_(...).addcmul_(..., ..., value=...)

                # Коррекция смещения, Q_t и масштаб s_t.
                mean = ...
                variance = ...
                direction = ...
                factor = ...

                # На первом шаге ограничения нет; далее рост не больше 1.01.
                update_norm = factor * gradient.norm()
                if "previous_norm" in state:
                    ratio = update_norm / (state["previous_norm"] + 1e-8)
                    correction = torch.clamp(ratio / 1.01, min=1.0)
                    factor = factor / correction
                    update_norm = update_norm / correction
                state["previous_norm"] = update_norm.detach()

                # Weight decay применяется к весам до основного обновления.
                parameter.mul_(1 - group["lr"] * group["weight_decay"])
                parameter.add_(..., alpha=-group["lr"] * math.sqrt(group["scale"]))
        return loss

check_apollo_mini(ApolloMini)
print("APOLLO-Mini: проверки пройдены")

**Вопрос:** Какую часть памяти этот метод обещает уменьшить,
а какую — нет? Почему AdamW для embedding и норм остаётся заметным
даже при ранге 1?

<!-- ответ -->

## 5. Четыре оптимизатора

Обучите модель четырьмя оптимизаторами. Каждый запуск — 180 шагов
`Trainer` с микробатчем 4 и накоплением по 16 микробатчам
(`gradient_accumulation_steps=16`): эффективный батч содержит 64 документа.
Исходные веса, seed, порядок документов и
валидационный набор одинаковы. Это около двух миллионов таргет-токенов
на метод. Ориентир для всего ноутбука — 45–60 минут GPU-времени на T4;
загрузка данных и заполнение упражнений в него не входят.

Learning rate уже выбраны:

| Метод | LR матричных весов | LR остальных весов |
|---|---:|---:|
| AdamW | $5\cdot10^{-5}$ | $5\cdot10^{-5}$ |
| Muon | $5\cdot10^{-4}$ | $5\cdot10^{-5}$ |
| Adam-mini | $5\cdot10^{-5}$ | $5\cdot10^{-5}$ |
| APOLLO-Mini | $2\cdot10^{-4}$ | $5\cdot10^{-5}$ |

Сравниваем методы при этих настройках. Подбирать LR или батч
в задании не требуется. Четыре основных запуска идут без профайлера.

In [ ]:
RUN_TAG = "steps180_first"  # После изменения кода задайте новое значение.
kinds = ("adamw", "muon", "adam_mini", "apollo")
runs = {
    kind: run_experiment(
        f"{kind}_{RUN_TAG}", kind,
        new_model=new_model, train_data=train_data, eval_data=eval_data,
        collate=collate_causal,
        split_parameters=split_parameters, apollo_class=ApolloMini,
        root=ROOT, steps=TOTAL_STEPS, device=device,
    )
    for kind in kinds
}

На первом графике по оси X — число таргет-токенов, на втором — время
обучения; по оси Y в обоих случаях — перплексия на валидации.
Каждая точка — реальное измерение; линии только соединяют их.

Время шага измеряется после разогрева GPU. Время обучения не включает
валидацию и сохранение диагностики.

Память по источникам снимается отдельно, в конце forward микробатча
из четырёх документов по 256 токенов, с уже выделенными буферами
градиентов. Каждая CUDA-аллокация относится ровно к одной категории:
веса и buffers, градиенты, optimizer state, входной батч, тензоры для
backward или прочее. Общие storage, в том числе tied weights, считаются
один раз. Эти части **можно складывать**: сумма равна `memory_allocated`
в момент снимка. Свободная память в кэше allocator в сумму не входит.
Здесь учитываются размеры аллокаций, включая округление allocator,
а не только число элементов тензоров.

Это снимок одного момента, **не разложение пика обучения**. Пик CUDA
показан отдельно. На последнем графике — медиана времени шага после
разогрева GPU.

Память состояния оптимизатора суммируется по всем группам параметров.

In [ ]:
# @title Техническая ячейка: таблица и графики
def read_run(path):
    summary = json.loads((path / "summary.json").read_text())
    history = [json.loads(line) for line in (path / "history.jsonl").read_text().splitlines()]
    return summary, history

rows = []
for kind, path in runs.items():
    summary, history = read_run(path)
    final_eval = [row for row in history if row["record_type"] == "eval"][-1]
    meter = summary["measurement"]
    rows.append({
        "method": kind, "таргет-токены": final_eval["target_tokens"],
        "лосс": final_eval["eval_loss"], "перплексия": math.exp(final_eval["eval_loss"]),
        "время, мин": final_eval["train_wall_time_seconds"] / 60,
        "шаг, с": meter["median_device_step_seconds"],
        "таргет-токены/с": meter["target_tokens_per_second"],
        "веса, ГиБ": meter["parameter_bytes"] / 2**30,
        "градиенты, ГиБ": meter["gradient_buffer_bytes"] / 2**30,
        "состояние, ГиБ": meter["optimizer_state_bytes"] / 2**30,
        "активации, ГиБ": meter["saved_for_backward_bytes"] / 2**30,
        "пик CUDA, ГиБ": meter["peak_cuda_allocated_bytes"] / 2**30,
        "загрузка GPU, %": meter["mean_gpu_utilization_percent"],
    })
comparison = pd.DataFrame(rows).set_index("method")
display(comparison.round(3))

histories = {kind: read_run(path)[1] for kind, path in runs.items()}
memory_snapshots = {kind: json.loads((path / "memory.json").read_text())
                    for kind, path in runs.items() if (path / "memory.json").exists()}
if memory_snapshots:
    memory_table = pd.DataFrame({kind: row["components_bytes"]
                                 for kind, row in memory_snapshots.items()}).T / 2**30
    memory_table["сумма, ГиБ"] = memory_table.sum(axis=1)
    display(memory_table.round(3))
for figure in (plot_quality(histories), plot_memory(comparison, memory_snapshots)):
    display(figure)
    plt.close(figure)

**Анализ:**

1. Какой метод получил лучший лосс при одинаковом числе таргет-токенов?
2. Сколько памяти состояния и общего пика сэкономил APOLLO-Mini
   относительно AdamW? Сопоставимо ли качество?
3. Как отличаются время шага и загрузка GPU? Кто быстрее достигает
   выбранного вами уровня перплексии?
4. Почему состояние Muon занимает не вдвое меньше памяти, чем состояние
   AdamW, хотя Muon хранит один momentum вместо двух моментов?
   Посчитайте вклад каждой группы параметров.

<!-- ответ -->

## 6. Геометрия шага Muon и Newton–Schulz

Muon преобразует матричное направление шага с помощью Newton–Schulz.
Разберём точный полярный фактор, а затем сравним его с результатом
нескольких итераций на матрице из нашего запуска.

> **Задача. Полярный фактор**
>
> Пусть $H\in\mathbb R^{n\times n}$ и $H=U\Sigma V^\top$ — её SVD.
> Докажите, что
>
> $$
> UV^\top\in\underset{R\in\mathbb R^{n\times n},\;R^\top R=I}
> {\operatorname{argmin}}\;\|H-R\|_F.
> $$
>
Для квадратной матрицы $UV^\top$ ортогональна. Если $H$ невырождена,
ближайшая ортогональная матрица единственна; при неполном ранге
единственность может нарушаться.

**Вопрос:** <!-- ответ -->

### Функции для анализа матриц

Пусть $\sigma_1\ge\cdots\ge\sigma_k$ — сингулярные значения матрицы
$X$. Реализуйте полярный фактор и следующие величины:

$$
\text{нормированный спектр:}\qquad
\left(\frac{\sigma_1}{\sigma_1},\ldots,\frac{\sigma_k}{\sigma_1}\right).
$$

$$
\operatorname{srank}(X)
=\frac{\|X\|_F^2}{\|X\|_2^2}
=\frac{\sum_{i=1}^k\sigma_i^2}{\sigma_1^2}.
$$

$$
r_{0.1}(X)=\#\{i:\sigma_i\ge0.1\sigma_1\}.
$$

$$
\operatorname{cos}(A,B)
=\frac{\langle A,B\rangle}{\|A\|_F\|B\|_F}.
$$

$$
\operatorname{dist}(A,B)=\frac{\|A-B\|_F}{\|B\|_F}.
$$

Stable rank может быть дробным, а $r_{0.1}$ считает значения выше
выбранного порога; обе величины отличаются от алгебраического ранга.
Для SVD используйте `torch.linalg.svd(..., full_matrices=False)`,
для одних сингулярных значений — `torch.linalg.svdvals`; вычисляйте их
в FP32. Функция `r01` принимает уже нормированный спектр.
Нулевой матрице сопоставьте нулевые спектр, stable rank и
$r_{0.1}$. Косинус и расстояние считаем только для ненулевых матриц.

In [ ]:
def polar_factor(matrix: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError

def normalized_singular_values(matrix: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError

def stable_rank(matrix: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError

def r01(spectrum: torch.Tensor) -> int:
    raise NotImplementedError

def matrix_cosine(left: torch.Tensor, right: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError

def relative_frobenius_distance(left: torch.Tensor, right: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError

check_geometry(polar_factor, normalized_singular_values, stable_rank, r01,
               matrix_cosine, relative_frobenius_distance)
print("Геометрия: проверки пройдены")

#### Ниже лишь анализ. Это была последняя ячейка с вашим кодом 🙂

Следующие две функции восстанавливают направление шага из сохранённых
данных. В `muon_direction` число итераций Newton–Schulz передаётся
аргументом; `adamw_direction` извлекает направление AdamW из его
моментов. Оба направления рассматриваем без
learning rate и weight decay.

In [ ]:
def muon_direction(matrix: torch.Tensor, ns_steps: int) -> torch.Tensor:
    candidate = nn.Parameter(torch.zeros_like(matrix, dtype=torch.float32))
    candidate.grad = matrix.detach().float().clone()
    optimizer = Muon(
        [candidate], lr=1.0, momentum=0.0, nesterov=False,
        weight_decay=0.0, ns_steps=ns_steps, adjust_lr_fn="original",
    )
    optimizer.step()
    ratio = math.sqrt(max(1.0, matrix.shape[0] / matrix.shape[1]))
    return -candidate.detach() / ratio

def adamw_direction(snapshot):
    beta1, beta2 = snapshot["betas"]
    step = snapshot["state_step"]
    mean = snapshot["exp_avg"] / (1 - beta1**step)
    variance = snapshot["exp_avg_sq"] / (1 - beta2**step)
    return mean / (variance.sqrt() + snapshot["eps"])

### Численный эксперимент

Для `q_proj` слоя 14 сравните 1–5 итераций Newton–Schulz с точным
полярным фактором, найденным через SVD. Вход Muon — градиент после
momentum/Nesterov; вычисления NS выполняются в FP16. Сверху показаны
относительное расстояние и cosine, снизу — спектры до и после NS.
Используемый Muon полином не обязан точно воспроизводить SVD.

Далее сравните **update** Muon и AdamW без learning rate и weight decay.
Это направление обновления, не веса и не исходный градиент. На тепловых
картах строки — сохранённые шаги, столбцы — номера сингулярных значений
в порядке убывания; цвет — $\sigma_i/\sigma_1$ по общей логарифмической
шкале. Строки показаны без интерполяции.

In [ ]:
# @title Техническая ячейка: таблица и графики
muon_snapshots = torch.load(runs["muon"] / "geometry.pt", map_location="cpu", weights_only=True)["snapshots"]
adam_snapshots = torch.load(runs["adamw"] / "geometry.pt", map_location="cpu", weights_only=True)["snapshots"]
chosen = min(muon_snapshots, key=lambda item: abs(item["step"] - TOTAL_STEPS // 2))
H = chosen["input"]
P = polar_factor(H)
ns_rows = []
spectra = {"H": torch.linalg.svdvals(H.float()).numpy(),
           "P": torch.linalg.svdvals(P.float()).numpy()}
for iterations in range(1, 6):
    direction = muon_direction(H, iterations)
    spectra[f"NS-{iterations}"] = torch.linalg.svdvals(direction.float()).numpy()
    ns_rows.append({
        "итераций NS": iterations,
        "относительное расстояние до P": float(relative_frobenius_distance(direction, P)),
        "cosine с P": float(matrix_cosine(direction, P)),
        "stable rank": float(stable_rank(direction)),
        "r01": r01(normalized_singular_values(direction)),
    })
ns_table = pd.DataFrame(ns_rows)
display(ns_table.rename(columns={
    "относительное расстояние до P": "относительное расстояние до полярного фактора",
    "cosine с P": "cosine с полярным фактором",
}).round(4))
figure = plot_newton_schulz(ns_table, spectra, step=chosen["step"])
display(figure)
plt.close(figure)

In [ ]:
# @title Техническая ячейка: таблица и графики
geometry_rows = []
for method, snapshots in [("Muon", muon_snapshots), ("AdamW", adam_snapshots)]:
    for snapshot in snapshots:
        direction = (muon_direction(snapshot["input"], NS_STEPS)
                     if method == "Muon" else adamw_direction(snapshot))
        spectrum = normalized_singular_values(direction)
        geometry_rows.append({"method": method, "step": snapshot["step"],
                              "stable rank": float(stable_rank(direction)),
                              "r01": r01(spectrum), "spectrum": spectrum.numpy()})
geometry_table = pd.DataFrame(geometry_rows)
display(geometry_table.drop(columns="spectrum").round(2))
figure = plot_geometry(geometry_table)
display(figure)
plt.close(figure)

**Анализ:**

1. Как меняются расстояние и косинус с полярным фактором по итерациям
   Newton–Schulz? Видно ли плато за пять итераций?
2. Как отличаются спектры, stable rank и $r_{0.1}$ у Muon и AdamW?
3. Что можно узнать о качестве модели по направлению шага одного слоя?

<!-- ответ -->

## 7. Gradient checkpointing

Два коротких запуска APOLLO-Mini отличаются только gradient
checkpointing. Это **проверка памяти и времени**, а не сравнение
качества: 16 шагов слишком мало для такого вывода. Сравните общий пик
CUDA, оценку активаций, сохранённых для обратного прохода, и время шага.
На графиках по оси X — режим checkpointing, по оси Y — соответствующее
измерение из таблицы.

In [ ]:
# @title Техническая ячейка: таблица и графики
probes = {
    state: run_experiment(
        f"apollo_checkpoint_{state}_{RUN_TAG}", "apollo",
        new_model=new_model, train_data=train_data, eval_data=eval_data,
        collate=collate_causal,
        split_parameters=split_parameters, apollo_class=ApolloMini,
        root=ROOT, steps=16, checkpointing=(state == "on"), device=device,
    )
    for state in ("off", "on")
}
probe_rows = []
for state, path in probes.items():
    summary, _ = read_run(path)
    meter = summary["measurement"]
    probe_rows.append({"checkpointing": state,
                       "пик CUDA, ГиБ": meter["peak_cuda_allocated_bytes"] / 2**30,
                       "активации, ГиБ": meter["saved_for_backward_bytes"] / 2**30,
                       "шаг, с": meter["median_device_step_seconds"]})
probe_table = pd.DataFrame(probe_rows).set_index("checkpointing")
display(probe_table.round(3))
figure = plot_checkpointing(probe_table)
display(figure)
plt.close(figure)

**Анализ:**

1. На сколько изменились пик памяти, сохранённые активации и время шага?
2. Объясните, почему checkpointing даёт такой обмен памяти на время.
3. Почему разность оценок активаций может отличаться от разности пиков?

<!-- ответ -->

## 8. Профайлер: таблица и временная диаграмма

`torch.profiler` запишет один шаг после разогрева GPU. Это отдельный короткий
запуск: его время не сравнивайте со временем четырёх обычных запусков.
Сначала посмотрите сводную таблицу. Строки
`ProfilerStep`, `HW02/...` и `Optimizer.step` охватывают вложенные
операции; их время нельзя складывать со строками `aten::...`.

In [ ]:
profile_run = run_experiment(
    f"apollo_profile_{RUN_TAG}", "apollo",
    new_model=new_model, train_data=train_data, eval_data=eval_data,
    collate=collate_causal,
    split_parameters=split_parameters, apollo_class=ApolloMini,
    root=ROOT, steps=4, profile=True, device=device,
)
print((profile_run / "profile.txt").read_text())

Для подробного разбора скачайте `profile.json.gz` и откройте его в
[Perfetto UI](https://ui.perfetto.dev/). Там можно приблизить один шаг
и одновременно видеть дорожки CPU, запуск операций CUDA и работу GPU —
как на временной диаграмме из семинара. Найдите большой промежуток без
операций GPU и проверьте, что в это время видно на дорожке CPU.
Посмотрите, есть ли рядом копирование данных или синхронизация.
Цветной интервал `HW02/microbatch` сам по себе не доказывает причину
простоя.

Вставьте в ноутбук **скриншот интересного вам фрагмента** из Perfetto.
На нём должны быть видны масштаб времени и дорожки, на которые вы
опираетесь. Подпишите выбранный участок и объясните, что он показывает.

In [ ]:
# В Colab раскомментируйте строки, чтобы скачать запись для Perfetto.
# from google.colab import files
# files.download(str(profile_run / "profile.json.gz"))

**Анализ:**

1. Какие две операции заняли больше всего времени GPU?
2. Что происходит в выбранном фрагменте на CPU и GPU? Если это простой,
   какие события рядом с ним помогают объяснить причину?
3. Какое измерение проверило бы вашу гипотезу о причине простоя?

<!-- ответ -->

## Итог

**Вывод:** Сформулируйте итог в терминах **качества, памяти и времени** для
проведённого протокола. Укажите хотя бы два ограничения эксперимента:
например, один seed и заранее выбранные LR. Не подменяйте близость
качества меньшим размером состояния оптимизатора.

<!-- ответ -->